In [1]:
##INSTALAMOS LAS DEPENDENCIAS
!pip install pymongo[srv] pandas python-dotenv

In [9]:
import pandas as pd
from pymongo import MongoClient
from datetime import datetime
from dotenv import load_dotenv
import os

load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")

client = MongoClient(MONGO_URI)
db = client["proyecto2_musica"]
coleccion = db["canciones"]

print("Conectado a MongoDB Atlas!")
print(f"Base de datos: {db.name}")

Conectado a MongoDB Atlas!
Base de datos: proyecto2_musica


In [3]:
##CARGAMOS EL CSV DEL PROYECTO 1
df = pd.read_csv("../data/raw/dataset_master.csv")
print(f" CSV cargado: {df.shape[0]} canciones, {df.shape[1]} columnas")
print(df.columns.tolist())

 CSV cargado: 10000 canciones, 44 columnas
['unnamed: 0', 'artist', 'track_name', 'year_original', 'genre', 'lyric', 'len', 'dating', 'violence', 'world/life', 'night/time', 'shake the audience', 'family/gospel', 'romantic', 'communication', 'obscene', 'music', 'movement/places', 'light/visual perceptions', 'family/spiritual', 'like/girls', 'sadness', 'feelings', 'danceability', 'loudness', 'acousticness', 'instrumentalness', 'valence', 'energy', 'topic', 'age', 'year', 'decade', 'lyric_clean', 'verbos', 'sustantivos', 'adjetivos', 'pronombres_count', 'densidad_lexica', 'ratio_sust_verb', 'ratio_adjetivos', 'palabras_clave', 'adjetivos_ejemplo', 'idioma_detectado']


In [4]:
##MIGRAR Y REORGANIZAR A MONGODB
# LIMPIAR
df = df.drop(columns=["unnamed: 0"])
df = df.where(pd.notnull(df), None)  # reemplaza NaN por None para MongoDB

# CONVERTIR
documentos = []
for _, row in df.iterrows():
    doc = {
        "titulo": row["track_name"],
        "artista": row["artist"],
        "genero": row["genre"],
        "anio": row["year"],
        "decada": row["decade"],
        "idioma": row["idioma_detectado"],
        "letra": row["lyric"],
        "letra_limpia": row["lyric_clean"],
        "fuente": "kaggle",
        "fecha_recopilacion": datetime.now(),
        "metricas_audio": {
            "danceability": row["danceability"],
            "loudness": row["loudness"],
            "acousticness": row["acousticness"],
            "instrumentalness": row["instrumentalness"],
            "valence": row["valence"],
            "energy": row["energy"],
        },
        "metricas_emocionales": {
            "dating": row["dating"],
            "violence": row["violence"],
            "romantic": row["romantic"],
            "sadness": row["sadness"],
            "feelings": row["feelings"],
        },
        "pos_tags": {
            "verbos": row["verbos"],
            "sustantivos": row["sustantivos"],
            "adjetivos": row["adjetivos"],
            "pronombres": row["pronombres_count"],
            "densidad_lexica": row["densidad_lexica"],
            "ratio_sust_verb": row["ratio_sust_verb"],
            "ratio_adjetivos": row["ratio_adjetivos"],
        },
        "embeddings": {
            "word2vec_avg": None,
            "beto_cls": None
        }
    }
    documentos.append(doc)pip install certifi dnspython

print(f" {len(documentos)} Documentos Listos")

 10000 Documentos Listos


In [6]:
##AGREGAR A MONGODB
# Limpiar colección si ya tiene datos
coleccion.delete_many({"fuente": "kaggle"})

# Insertar en lotes de 1000
lote = 1000
for i in range(0, len(documentos), lote):
    coleccion.insert_many(documentos[i:i+lote])
    print(f" DATOS Insertados {min(i+lote, len(documentos))}/{len(documentos)}")

print(f"\n Migración completa! Total en DB: {coleccion.count_documents({})}")

ServerSelectionTimeoutError: SSL handshake failed: ac-c1alydh-shard-00-02.xdk6hlm.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1000) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-c1alydh-shard-00-01.xdk6hlm.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1000) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-c1alydh-shard-00-00.xdk6hlm.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1000) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 69c557816f670e0226e45f71, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('ac-c1alydh-shard-00-00.xdk6hlm.mongodb.net', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('SSL handshake failed: ac-c1alydh

In [10]:
##VERIFICAR QUE ESTEN LOS DATOS
#GENEROS
print("Canciones por género:")
pipeline = [{"$group": {"_id": "$genero", "total": {"$sum": 1}}}]
for doc in coleccion.aggregate(pipeline):
    print(f"  {doc['_id']}: {doc['total']}")

#MUESTRA
print("\n Muestra de documento:")
muestra = coleccion.find_one({"genero": "pop"})
print(f"  Título: {muestra['titulo']}")
print(f"  Artista: {muestra['artista']}")
print(f"  Género: {muestra['genero']}")
print(f"  Idioma: {muestra['idioma']}")
print(f"  Densidad léxica: {muestra['pos_tags']['densidad_lexica']}")


Canciones por género:


ServerSelectionTimeoutError: SSL handshake failed: ac-c1alydh-shard-00-02.xdk6hlm.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1000) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-c1alydh-shard-00-01.xdk6hlm.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1000) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-c1alydh-shard-00-00.xdk6hlm.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1000) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 69c559106f670e0226e45f74, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('ac-c1alydh-shard-00-00.xdk6hlm.mongodb.net', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('SSL handshake failed: ac-c1alydh

In [11]:
##CERRAMOS CONEXION
client.close()
print("Conexión cerrada")

Conexión cerrada
